# RAG：Knowledge Base 與 Hybrid Retrieval 建置

將前一階段完成之 LLM 標註資料作為 Knowledge Base，利用 OpenAI Embedding 建立向量表示，並結合 FAISS 與 Metadata Filtering 建構 Hybrid Retrieval

此外，透過 Retrieval Benchmark 比較 Pure Vector Search 與 Hybrid Retrieval 的檢索品質，驗證 Metadata Filtering 是否能有效提升語意搜尋結果，作為後續 Agent 的知識檢索核心

In [ ]:
# 1.dataset
from google.colab import drive
import pandas as pd
import numpy as np
import os
drive.mount('/content/drive')
df_llm = pd.read_csv("/content/drive/MyDrive/colabnotebooks/LLM_annotation_result.csv", encoding='big5')

In [ ]:
# 2.build knowlegebase
knowledge_base = (df_llm[["member","song","views","comment","focus_topic","LLM_sentiment","rewritten_sentence"]].dropna(subset=["rewritten_sentence"]).reset_index(drop=True))

In [ ]:
# 3.openai embedding
!pip install -q openai tqdm

from openai import OpenAI
from tqdm import tqdm

try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY").strip()

except Exception:
    api_key = os.environ.get("OPENAI_API_KEY").strip()

client = OpenAI(api_key=api_key)
EMBEDDING_MODEL = "text-embedding-3-small"

def get_embedding(text):
    response = client.embeddings.create(model=EMBEDDING_MODEL,input=text)
    return response.data[0].embedding

**| 建立向量索引**

embedding 將留言轉換為語意向量，並建立 FAISS Index 作為向量資料庫，使系統能依語意相似度搜尋相關留言，而非僅依關鍵字比對。

In [ ]:
# 4.Embedding(rewritten sentence)
import numpy as np
from tqdm import tqdm

embeddings = []
for text in tqdm(knowledge_base["rewritten_sentence"]):
    embeddings.append(get_embedding(text))
embedding_matrix = np.array(embeddings,dtype=np.float32)
np.save("embeddings.npy", embedding_matrix)
print(embedding_matrix.shape)

In [ ]:
# 5. build FAISS index
!pip install -q faiss-cpu
import faiss

embedding_matrix = np.load("embeddings.npy").astype("float32")
faiss.normalize_L2(embedding_matrix)

dimension = embedding_matrix.shape[1]

global_index = faiss.IndexFlatIP(dimension)
global_index.add(embedding_matrix)

**| 建立雙引擎查詢**

除了語意檢索外，額外建置 Structured Analytics，利用 Pandas 直接處理平均值、數量、排序等結構化查詢

*   當使用者提出**統計**問題時，系統直接呼叫結構化分析引擎清單項目
*   若屬**留言語意理解、其他資料搜尋**，則進入 Hybrid Retrieval，以兼顧資料分析與自然語言問答能力




In [ ]:
# 6.structured engine
AGG_FUNC = {"mean":"mean","max":"max","min":"min"}
def pandas_analytics(parsed):
  df = knowledge_base.copy()
  if parsed.member:df = df[df["member"] == parsed.member]
  if parsed.song:df = df[df["song"] == parsed.song]
  if parsed.focus_topic:df = df[df["focus_topic"] == parsed.focus_topic]
  if parsed.group_by is None:
    if parsed.aggregation == "count":
        return len(df)
  if parsed.aggregation == "count":
    return (df.groupby(parsed.group_by).size().reset_index(name="count").sort_values("count", ascending=False))
  else:
    result = (df.groupby(parsed.group_by)[parsed.metric].agg(AGG_FUNC[parsed.aggregation]).reset_index().sort_values(parsed.metric, ascending=False))
  return result

In [ ]:
def metadata_filter(parsed):
    candidate = knowledge_base.copy()
    filters = parsed.model_dump(
    exclude_none=True,
    exclude={"question","query_type","aggregation","metric","group_by"})
    for col, value in filters.items():
        # 只有 df 有這個欄位才 filter
        if col not in candidate.columns:
            continue
        if isinstance(value,str):
            candidate = candidate[
                candidate[col]
                .astype(str)
                .str.strip()
                .str.lower()
                ==
                value.strip().lower()
            ]
        else:
            candidate = candidate[candidate[col] == value]
    return candidate

In [ ]:
def hybrid_search(parsed, candidate, k=5):
    query_vector = get_embedding(parsed.question)
    query_embedding = np.array([query_vector],dtype=np.float32)
    faiss.normalize_L2(query_embedding)

    candidate_idx = candidate.index.to_numpy()
    candidate_embeddings = embedding_matrix[candidate_idx]
    scores = np.dot(query_embedding,candidate_embeddings.T)[0]

    top_idx = np.argsort(scores)[::-1][:k]
    results = candidate.iloc[top_idx].copy()
    results["similarity"] = scores[top_idx]
    return results

In [ ]:
def pure_vector_search(query, k=5):
    query_vector = get_embedding(query)
    query_embedding = np.array([query_vector], dtype=np.float32)
    faiss.normalize_L2(query_embedding)

    # Global FAISS
    distances, indices = global_index.search(query_embedding, k)

    results = knowledge_base.iloc[indices[0]].copy()
    results["similarity"] = distances[0]
    return results

In [ ]:
# schema
from pydantic import BaseModel
from typing import Literal
from typing import Optional

class ParsedQuery(BaseModel):
    question: str

    query_type: Literal["semantic","structured"]
    member: Optional[str] = None
    song: Optional[str] = None
    focus_topic: Optional[str] = None

    aggregation: Optional[Literal["count","mean","max","min"]] = None
    metric: Optional[Literal["views","LLM_sentiment","focus_topic"]] = None
    group_by: Optional[Literal["member","song","focus_topic"]] = None

**| Query Parser**

LLM 解析使用者問題，自動判斷查詢屬於：

- Structured Query（資料分析）
- Semantic Query（留言理解）

並同步抽取 Member、Song、Focus Topic、LLM_sentiment 等 Metadata，作為後續工具調度依據

In [ ]:
def parse_query(query):
  system_prompt = """你是一位 Query Parser。你的工作是分析使用者問題，並將其解析成結構化 JSON。
Step 1. 判斷 Query Type

semantic：
需要閱讀留言內容才能回答。

例如：
- Hitomi 的舞蹈如何？
- Panorama 大家都在討論什麼？
- Wonyoung 的外貌評價？
- Sakura 的舞蹈實力如何？
- Yuri 的唱功好嗎？

structured：
若問題可直接透過資料表統計、排序、聚合即可回答，不需要閱讀留言內容。

例如：
- 哪位成員觀看數最高？
- Fiesta 有多少留言？
- 平均觀看數最高是哪首歌？
- 哪首歌留言最多？
- Chaeyeon 的留言以哪個 focus topic 為主？

Step 2. Metadata Parsing
請抽取：member,song，focus_topic

若沒有提及請填 null。

focus_topic 僅能為：visual,performance,vocal,identity,viral

Step 3. Structured Query Parsing
若 query_type = structured，請額外解析：
aggregation：count,mean,max,min
metric：views,LLM_sentiment,focus_topic
group_by：member,song,focus_topic

若沒有需要請填 null。

Step 4. Parsing Rules

【Rule A：Count Query】

若問題是在詢問：

- 哪個最多
- 哪個最常出現
- 哪個為主
- 哪個為大宗
- 哪位留言最多
- 哪首歌留言最多
- 哪個 topic 最多

代表需要統計每個類別的筆數。

請解析為：
aggregation = count
metric = null
group_by = 對應欄位

例如：

Q：哪位成員留言最多？
aggregation = count
metric = null
group_by = member

Q：哪首歌留言最多？
aggregation = count
metric = null
group_by = song

Q：Chaeyeon 的留言以哪個 focus topic 為主？
aggregation = count
metric = null
group_by = focus_topic
member = chaeyeon


【Rule B：Aggregation Query】

若問題是在詢問：

平均
最高
最低
最大值
最小值

且目標是數值欄位，

請解析：
aggregation = mean / max / min
metric = views 或 LLM_sentiment

例如：
Q：哪位成員平均觀看數最高？
aggregation = mean
metric = views
group_by = member

Q：哪首歌觀看數最高？
aggregation = max
metric = views
group_by = song

【Rule C：focus_topic】
focus_topic 為類別(Label)，不是數值！
因此：不得解析
aggregation = max
metric = focus_topic

若詢問：
- 哪個 topic 最多
- 哪個 topic 為主
- 哪個 topic 為大宗
皆應解析為：

aggregation = count
metric = null
group_by = focus_topic


Step 5. Semantic Query

若 query_type = semantic，question 保留真正需要語意搜尋的內容。


請輸出符合 ParsedQuery 的 JSON。不要輸出任何額外說明。
"""
  response = client.beta.chat.completions.parse(
        model="gpt-4.1-mini",
        messages=[
            {"role":"system","content":system_prompt},
            {"role":"user","content":query}
        ],
        response_format=ParsedQuery,
        temperature=0
    )
  return response.choices[0].message.parsed

In [ ]:
import pandas as pd
def response_generator(output,query):
    result_type = output["type"]
    result = output["result"]
    source = output.get("source",None)
    # Structured Query (Pandas)
    if result_type == "structured":
        table = result.to_markdown(index=False)
        prompt = f"""你是一位資料分析助手。以下是 Pandas 查詢結果：

{table}

使用者問題：

{query}

回答規則：

1. 僅根據表格回答。
2. 請針對問題整理成精簡扼要的自然語言，不用印出所有結果
3. 若有排序，請說明第一名!
4. 若有多筆資料，可比較。
5. 不可使用外部知識。
6. 去除「 」、** 的符號輸出，改為加深粗體
"""
        response = client.chat.completions.create(model="gpt-4.1-mini",messages=[{"role":"system","content":"你是一位資料分析助手。"},{"role":"user","content":prompt}],temperature=0)
        return response.choices[0].message.content

    # Semantic Query (Hybrid RAG)
    elif result_type == "semantic":
        context = ""
        for rank, (_, row) in enumerate(result.iterrows(), start=1):
            context += f"""
[Reference {rank}]

Member: {row.member}
Song: {row.song}
Views: {row.views}
Topic: {row.focus_topic}
Sentiment: {row.LLM_sentiment}
Comment:{row.rewritten_sentence}
"""
 # Pure Vector fallback 提醒
        if source == "pure_vector":
            retrieval_instruction = """
注意：
本次回答來自 Pure Vector 語意相似檢索，
並非完全符合 Metadata 條件。

請：
1. 不要將檢索結果視為問題對象的直接證據。
2. 若留言資訊與問題不完全符合，請明確說明限制。
3. 避免過度推論。
"""
        else:
            retrieval_instruction = """
本次回答來自 Hybrid Retrieval。
Metadata 條件與語意相似度皆已考量，
可根據檢索留言進行摘要。
"""
        prompt = f"""你是一位 K-pop 留言分析助手。以下為從知識庫檢索出的留言。

{context}

請回答：

{query}

回答規則：

1. 僅能根據留言回答。
2. 若至少一則留言可以回答問題，就請整理精簡扼要回答。
3. 若多則留言觀點一致，請整合成摘要。
4. 若留言完全沒有相關資訊，再回答：目前知識庫中沒有足夠資訊回答此問題。
5. 不可使用外部知識。
6. 去除「 」、** 的符號輸出，改為加深粗體
"""
        response = client.chat.completions.create(model="gpt-4.1-mini",messages=[{"role":"system","content":"你是一位資料分析助手。"},{"role":"user","content":prompt}],temperature=0)
        return response.choices[0].message.content
    else:
        return "無法處理此查詢類型。"

**| Agent workflow**

可依據查詢內容自動選擇資料分析或語意檢索流程，完成從問題解析、工具選擇、資訊檢索到自然語言回答的完整閉環

In [ ]:
# AI Agent
def AI_Agent(query):
    parsed = parse_query(query)
    if parsed.query_type == "structured":
        result = pandas_analytics(parsed)
        output = {"type": "structured","result": result,"source":"pandas"}
    elif parsed.query_type == "semantic":
        candidate = metadata_filter(parsed)
        if len(candidate) > 0:
            result = hybrid_search(parsed,candidate)
            output = {"type":"semantic","result":result,"source":"hybrid","candidate_size":len(candidate)}
        else:
            print("Metadata 無符合結果，改用 Pure Vector Search")
            result = pure_vector_search(parsed.question)
            output = {"type":"semantic","result":result,"source":"pure_vector","candidate_size":0}
    else:
      return "目前無法判斷查詢類型。"
    answer = response_generator(output, query)
    return answer

In [ ]:
query = input("請輸入問題：")
answer = AI_Agent(query)
print(answer)

**| Retrievel evaluation**

建立 Benchmark Query，比較 Pure Vector Search ( 0.89 )  與 Hybrid Retrieval ( 3.33 ) 的檢索品質。以 Relevant@5 作為評估指標，驗證 Metadata Filtering 對 Retrieval 品質之提升效果

因 Pure Vector Search 保留較高的查詢覆蓋率，可於 Metadata 無符合結果時作為補充檢索策略，使整體 Retrieval Pipeline 同時兼顧 Precis

In [ ]:
benchmark = [
    {
        "query": "沒想到 wonyoung 年紀這麼小？",
        "member": "wonyoung",
        "focus_topic": "identity"
    },
    {
        "query": "chaewon真的唱功最好嗎？",
        "member": "chaewon",
        "focus_topic": "vocal"
    },
    {
        "query":"大家如何評價 yuri 的唱功",
        "member":"yuri",
        "focus_topic":"vocal"
    },
    {
        "query":"sakura的舞蹈如何？",
        "member":"sakura",
        "focus_topic":"performance"
    },
    {
        "query":"yena的綜合實力想必是頂尖吧?",
        "member":"yena",
        "focus_topic":"performance"
    },
    {
        "query":"yujin 的表現力一值都如此出眾嗎?",
        "member":"yujin",
        "focus_topic":"performance"
    },
    {
        "query":"大眾如何評價 Nako 解散後的發展?",
        "member":"nako",
        "focus_topic":"identity"
    },
    {
        "query":"minju他真的有魅力嗎?",
        "member":"minju",
        "focus_topic":"identity"
    },
    {
        "query":"eunbi的外貌如何?",
        "member":"eunbi",
        "focus_topic":"visual"
    }
]

In [ ]:
def relevant_at_5(results, expected):
    score = 0
    for _, row in results.iterrows():
        match = True
        for key, value in expected.items():
            if key == "query":
                continue
            if row[key] != value:
                match = False
                break
        if match:
            score += 1
    return score

In [ ]:
import faiss
evaluation = []
for item in benchmark:
    query = item["query"]
    # Pure Vector
    pure_results = pure_vector_search(query)
    pure_score = relevant_at_5(pure_results,item)

    # Hybrid
    parsed = parse_query(query)
    candidate = metadata_filter(parsed)

    # fallback | fallback = False,if len(candidate) == 0:fallback = True,candidate = knowledge_base.copy()

    hybrid_results = hybrid_search(parsed,candidate)
    hybrid_score = relevant_at_5(hybrid_results,item)
    evaluation.append({
        "Query":query,
        "Pure Relevant@5":pure_score,
        "Hybrid Relevant@5":hybrid_score,
        "Candidate After":len(candidate)})

evaluation = pd.DataFrame(evaluation)
evaluation

In [ ]:
summary = pd.DataFrame({
    "Metric":["Average Relevant@5 (Pure)","Average Relevant@5 (Hybrid)","Average Candidate After"],
    "Value":[evaluation["Pure Relevant@5"].mean(),evaluation["Hybrid Relevant@5"].mean(),evaluation["Candidate After"].mean()]
})
summary

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(5,4))
plt.bar(["Pure Vector","Hybrid RAG"],
    [evaluation["Pure Relevant@5"].mean(),evaluation["Hybrid Relevant@5"].mean()])

plt.ylabel("Average Relevant@5")
plt.title("Retrieval Quality Comparison")
plt.show()

## 研究結論

本研究完成 Knowledge Base、Embedding、FAISS、Hybrid Retrieval 與 AI Agent 建置。建立雙引擎查詢架構，使系統可依使用者需求自動切換資料分析或語意檢索流程，完成從自然語言查詢、知識檢索到回答生成的AI Workflow